In [1]:
import os   

import pandas as pd
import numpy as np
import numpy as np
import xdas as xd
import seisbench.models as sbm

from obspy import read
from obspy import UTCDateTime
from geopy.distance import geodesic
from obspy.clients.fdsn import RoutingClient
from obspy.clients.fdsn import Client
client = RoutingClient("eida-routing")

In [2]:
def unify_picks(st, picks):
    df = pd.DataFrame({"station": [p.trace_id.split(".")[1] for p in picks.picks], "phase": [p.phase for p in picks.picks], "pick": [p.peak_time for p in picks.picks]})
    pp = df[df["phase"] == "P"].groupby("station", as_index=False)["pick"].min().rename(columns={"pick": "ppick"})
    sp = df[df["phase"] == "S"].groupby("station", as_index=False)["pick"].min().rename(columns={"pick": "spick"})
    stations = pd.DataFrame({"station": sorted(set([tr.stats.station for tr in st]))})
    df_picks = stations.merge(pp, on="station", how="left").merge(sp, on="station", how="left")
    df_picks["ppick"] = df_picks["ppick"].where(df_picks["ppick"].notnull(), "Nat")
    df_picks["spick"] = df_picks["spick"].where(df_picks["spick"].notnull(), "Nat")
    return df_picks


def evaluate_peak_disp(tr, pick_time, wlen=6, perc=98, input_type="velocity", lowcut=0.075, highcut=25):
    tr_disp = tr.copy()
    tr_disp.taper(max_percentage=0.05, type="hann")
    tr_disp.filter("bandpass", freqmin=lowcut, freqmax=highcut, corners=4, zerophase=True)
    if input_type == "velocity":
        tr_disp.integrate()  # integrate velocity to displacement
    elif input_type == "acceleration":
        tr_disp.integrate().integrate()  # integrate acceleration to displacement
    disp = tr_disp.data.astype(float)
    times = tr.times("utcdatetime")
    mask = (times >= pick_time) & (times < (pick_time + wlen))
    window = disp[mask]
    if window.size == 0:
        raise ValueError("Zero-length time window!")
    return float(np.nanpercentile(np.abs(window), perc))


def evaluate_peak_noise(tr, wlen=6, perc=98, input_type="velocity", lowcut=0.075, highcut=25):
    tr_disp = tr.copy()
    tr_disp.taper(max_percentage=0.05, type="hann")
    tr_disp.filter("bandpass", freqmin=lowcut, freqmax=highcut, corners=4, zerophase=True)
    if input_type == "velocity":
        tr_disp.integrate()  # integrate velocity to displacement
    elif input_type == "acceleration":
        tr_disp.integrate().integrate()  # integrate acceleration to displacement    noise = tr_noise.data.astype(float)
    times = tr.times("utcdatetime")
    noise = tr_disp.data.astype(float)
    times = tr.times("utcdatetime")
    start = tr.stats.starttime
    mask = (times >= start) & (times < (start + wlen))
    window = noise[mask]
    if window.size == 0:
        raise ValueError("Zero-length time window!")
    return float(np.nanpercentile(np.abs(window), perc))


def hypodist(tr, event_info):
    sta_lat = tr.stats.sac.stla
    sta_lon = tr.stats.sac.stlo
    event_lat = event_info["latitude"]
    event_lon = event_info["longitude"]
    event_depth_km = event_info["depth"]
    surface_dist_km = geodesic((sta_lat, sta_lon), (event_lat, event_lon)).km
    return np.sqrt(surface_dist_km**2 + event_depth_km**2)


def get_peak_ampl(st, df_picks, event_info, wlen=6, perc=98, input_type="velocity", phase="P"):
    results = []
    for tr in st:
        station_code = tr.stats.station
        phase_label = "ppick" if phase == "P" else "spick"
        pick_time = df_picks.loc[df_picks["station"] == station_code, phase_label].values[0]
        if pick_time == "Nat":
            continue  # Skip stations without picks
        pick_time = UTCDateTime(pick_time)
        try:
            peak_disp = evaluate_peak_disp(tr, pick_time, wlen=wlen, perc=perc, input_type=input_type, lowcut=0.075, highcut=25)
            peak_noise = evaluate_peak_noise(tr, wlen=wlen, perc=perc, input_type=input_type, lowcut=0.075, highcut=25)
            snr = peak_disp / peak_noise if peak_noise > 0 else np.inf
            distance = hypodist(tr, event_info)
            results.append(
                    {
                    "station": station_code, 
                    "peak_disp": peak_disp, 
                    "peak_noise": peak_noise, 
                    "snr": snr,
                    "distance": distance,
                    "latitude": tr.stats.sac.stla,
                    "longitude": tr.stats.sac.stlo,
                    "depth": tr.stats.sac.stel
                    }
                )
        except ValueError:
            continue  # Skip if no data in the window
    return pd.DataFrame(results)


def ampl2mag(amplitude_df, fit, phase="P", wlen=6, coeff=["slope", "k", "c", "a"], snr_thre=10):
    scaling_law_ps = dict(
        {coeff: fit[0][ix] for ix, coeff in enumerate(coeff)}
    )
    slope, k, c, a = scaling_law_ps["slope"], scaling_law_ps["k"], scaling_law_ps["c"], scaling_law_ps["a"]
    mask = amplitude_df["snr"] >= snr_thre
    magnitude_df = amplitude_df[mask].copy()
    magnitude_df["magnitude"] = (np.log10(magnitude_df["peak_disp"]) - slope * np.log10(magnitude_df["distance"]) - k * (magnitude_df["distance"] - 1.0) - c) / a
    return magnitude_df

# Catalogue and abyss infos


In [3]:
catalog = pd.read_csv("../CSN_bulletin_close.csv", parse_dates=["time"])
dc_slope = xd.open_datacollection(f'../fit_slope.nc')
dc_k = xd.open_datacollection(f'../fit_k.nc')
measurand = "displacement"
slope_ps, slope_s = -dc_slope[measurand]['PS'].values, -dc_slope[measurand]['S'].values
k_ps, k_s = -dc_k[measurand]['PS'].values, -dc_k[measurand]['S'].values
fit_ps2 = slope_ps, k_ps, -6.8, 0.75
fit_ps4 = slope_ps, k_ps, -6.9, 0.8
fit_ps6 = slope_ps, k_ps, -7.0, 0.84
fit_s = slope_s, k_s, -5.9, 1.0

In [4]:
catalog

,time,latitude,longitude,depth,mag,magType,event_id,min_hyp_CCN_N_km,min_hyp_SER_N_km,min_hyp_SER_S_km
0,2025-06-13 01:06:39,-32.671,-71.801,34.0,4.0,Ml,20250613T010639,34.3,314.3,192.8
1,2025-06-24 02:58:26,-29.352,-71.484,68.0,4.3,Ml,20250624T025826,267.7,72.3,91.5
2,2025-06-24 05:08:00,-31.613,-72.609,42.0,4.0,Ml,20250624T050800,85.6,231.8,109.5
3,2025-06-28 09:58:38,-29.752,-71.291,57.0,4.7,Mw,20250628T095838,225.8,58.6,59.0
4,2025-06-30 08:12:28,-31.960,-71.240,57.0,4.6,Mw,20250630T081228,81.4,236.6,138.2
...,...,...,...,...,...,...,...,...,...,...
76,2026-03-13 13:39:22,-28.780,-71.480,28.0,6.5,Mww,20260313T133922,323.2,58.9,127.2
77,2026-03-13 13:49:25,-28.746,-71.452,31.0,4.0,Mlv,20260313T134925,327.4,64.4,131.4
78,2026-03-14 03:31:55,-28.778,-71.321,33.0,4.0,Mlv,20260314T033155,325.7,69.0,127.6
79,2026-03-14 04:11:21,-28.828,-71.458,26.0,4.2,Mlv,20260314T041121,317.9,54.8,121.5


# Select event, download F1 data, pick with Phasenet

In [11]:
model = sbm.PhaseNet.from_pretrained("diting")

sel_event = "20260222T142257" # Event ID to download
event_info = catalog.loc[catalog["event_id"] == sel_event].iloc[0]
event_time = UTCDateTime(event_info["time"].to_pydatetime())
t0 = event_time - 10
t1 = event_time + 120
inventory = client.get_stations(
    network="F1",
    station=f"*",
    location="*",
    channel="*",
    starttime=t0,
    endtime=t1,
    level="response",
)
stream = client.get_waveforms(
    network="F1",
    station=f"*",
    location="*",
    channel="*",
    starttime=t0,
    endtime=t1,
)
start = max(tr.stats.starttime for tr in stream)
end   = min(tr.stats.endtime for tr in stream)
stream = stream.trim(start, end)
picks = model.classify(stream)
st = stream.copy()
for i, tr in enumerate(st):
    station_code = tr.stats.station
    tr.stats.sac = {
        "stla": inventory.select(network="F1", station=f"{station_code}")[0][0].latitude,
        "stlo": inventory.select(network="F1", station=f"{station_code}")[0][0].longitude,
        "stel": inventory.select(network="F1", station=f"{station_code}")[0][0].elevation,
    }
    tr.stats.channel = "HHZ"
    tr.data = tr.data * 1e-10  # convert from nm/s to m/s and remove the factor 10 from instrumental response


# Evaluate magnitude

In [12]:
phase = "P"
wlen = 6
if phase == "P":
    fit =[fit_ps2 if (wlen>=2) and (wlen<4) else fit_ps4 if (wlen>=4) and (wlen<6) else fit_ps6]
elif phase == "S":
    fit = [fit_s]
coeff = ["slope", "k", "c", "a"]
snr_thre = 10
df_picks = unify_picks(st, picks)
amplitude_df = get_peak_ampl(st, df_picks, event_info, wlen=wlen, perc=98, input_type="velocity", phase=phase)
magnitude_df = ampl2mag(amplitude_df, fit, phase=phase, wlen=wlen, coeff=coeff, snr_thre=snr_thre)
magnitude_df.to_csv(f"../results/magnitude_estimation_{sel_event}_{phase}_wlen{wlen}.csv", index=False)

In [13]:
print(f"Median magnitude: {np.nanmedian(magnitude_df['magnitude'].values):.1f} True magnitude: {event_info['mag']}")

Median magnitude: 4.9 True magnitude: 4.8
